In [15]:
import jax.numpy as jnp
import jax

embedding_dim = 1024
min_period = 1.0
max_period = 4.0 
pos = jnp.array([1.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5]).reshape(1, 7)
pos = jnp.squeeze(pos)
fraction = jnp.linspace(0.0, 1.0, embedding_dim)
period = min_period * (max_period / min_period) ** fraction
sinusoid_input = jnp.einsum(
    "i,j->ij",
    pos,
    1.0 / period * 2 * jnp.pi,
    precision=jax.lax.Precision.HIGHEST,
)

sinusoid_input
sinusoid_input.shape





(7, 1024)

In [10]:
import jax.numpy as jnp
arr = 1
arr = jnp.repeat(arr, 15, axis=-1).reshape(1,-1)
print(arr, arr.shape)

[[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]] (1, 15)


In [1]:
def plot_episode_history(joint_idx=6): 
    import zarr 
    from collections import deque
    import time 
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path

    ZARR_PATH = Path(f"data_vla/eval_runs/2026-04-02/episodes/RTC_NOGAUSS_15-44-07/").resolve()
    OBS_ZARR = ZARR_PATH / "obs_recorder.zarr"
    ROBOT_ZARR = ZARR_PATH / "robot_controller.zarr"
    try:
        print(f"[INFO] Åpner Zarr: {OBS_ZARR} ...")
        root = zarr.open(str(OBS_ZARR), mode="r")
    except Exception as e:
        print(f"[ERROR] Kunne ikke åpne Zarr: {e}")
        return
    try:
        print(f"[INFO] Åpner Zarr: {ROBOT_ZARR} ...")
        robot_root = zarr.open(str(ROBOT_ZARR), mode="r")
    except Exception as e:
        print(f"[ERROR] Kunne ikke åpne Zarr: {e}")
        return

    print("[INFO] Zarr-innhold:")
    print(root.tree())
    
    print("[INFO] Robot Zarr-innhold:")
    print(robot_root.tree())

    timess = np.array(root["observations"]["timestamps"]) * 1e-9 # Seconds 
    states = np.array(root["observations"]["robot_states"])
    
    plt.figure(figsize=(12, 6))
    plt.plot(timess, states[:, joint_idx], color=cpallete[1], label="Ground truth")
    
    for i, (t, state) in enumerate(zip(timess, states)):
        
        t_start = t
        
        actions = state[joint_idx] + np.array(robot_root["signals"]["action_chunks"]["values"][i, :, joint_idx])
        action_times = t_start + np.arange(actions.shape[0])*(1/15) # Assuming 15Hz action frequency

        plt.plot(action_times, actions, color=cpallete[9])
    
    plt.title(f"Episode history")
    plt.xlabel("Time (s)")
    plt.ylabel(f"Joint positions")
    plt.grid()
    plt.legend()

plot_episode_history(joint_idx=6)


[INFO] Åpner Zarr: /home/ril_mtplab/workspace/ril_franka-VLA/third_party/openpi/src/openpi/data_vla/eval_runs/2026-04-02/episodes/RTC_NOGAUSS_15-44-07/obs_recorder.zarr ...
[ERROR] Kunne ikke åpne Zarr: Unable to find group: file:///home/ril_mtplab/workspace/ril_franka-VLA/third_party/openpi/src/openpi/data_vla/eval_runs/2026-04-02/episodes/RTC_NOGAUSS_15-44-07/obs_recorder.zarr
